# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [3]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.1 MB/s eta 0:00:00


In [4]:
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


### **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [5]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = '¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?'

print(prompt)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [29]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

**RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (SSD/HDD, NVMe, etc.)**

| Característica | RAM | Almacenamiento |
|----------------|-----|----------------|
| **Volatilidad** | **Volátil**: pierde su contenido cuando la computadora se apaga. | **No volátil**: conserva los datos incluso sin energía. |
| **Propósito** | Mantener y procesar datos **en tiempo real** que la CPU necesita mientras la máquina está encendida. | Guardar datos **a largo plazo**: archivos del sistema, programas, fotos, videos, etc. |
| **Velocidad** | Muy alta (nanosegundos). | Más lenta (microsegundos a milisegundos), aunque los SSD modernos se acercan a velocidades de RAM. |
| **Capacidad** | Limitada (de 4 GB a 64 GB típicamente en ordenadores personales; servidores pueden tener 128 GB+). | Mucho mayor (de 128 GB a varios terabytes). |
| **Costo por GB** | Muy alto (varias veces más caro que el almacenamiento). | Bajo en comparación; el precio de los discos SSD ha bajado dramáticamente. |
| **Tipo de tec

In [62]:
models = client.models.list()

for model in models.data:
    if "llama" in model.id.lower():
        print(model.id)

meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m


In [69]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-4f55ec8e-b15d-468a-9a24-5f0626fa1a95",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "La **RAM** (memoria de acceso aleatorio) y el **almacenamiento** (disco duro, SSD, etc.) cumplen funciones distintas en una computadora, aunque ambos son tipos de memoria. A continuación se detalla cada uno y las diferencias clave:\n\n| Aspecto | RAM | Almacenamiento (HDD/SSD/Disco externo, etc.) |\n|---------|-----|----------------------------------------------|\n| **Tipo de memoria** | Volátil | No volátil |\n| **Permanencia** | Se pierde cuando se apaga o reinicia el equipo | Se conserva incluso sin energía |\n| **Velocidad** | Muy rápida (nanosegundos) | Más lenta (microsegundos a milisegundos) |\n| **Propósito principal** | Mantener datos y programas que el CPU necesita usar de inmediato | Guardar datos de forma permanente: sistema operativo, programas, archivos personales, etc. |\n| **Acceso

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [8]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)

Tokens del prompt: 86
Tokens de la respuesta: 635
Tokens totales: 721


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [30]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 1.56 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [31]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
inicio = time.time()
response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Modelo ligero: 1.56 s
Modelo grande: 2.05 s — 935 tokens

Respuesta del modelo grande:
 **RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (disco duro / SSD)**  

| Característica | RAM | Almacenamiento |
|----------------|-----|----------------|
| **Función principal** | Guardar datos e instrucciones que la CPU necesita **en el momento** para ejecutar programas. | Guardar de forma **permanente** todos tus archivos, sistemas operativos, aplicaciones, etc. |
| **Volatilidad** | **Volátil**: al apagar o reiniciar el equipo la información se pierde. | **No volátil**: los datos permanecen aunque el equipo esté apagado. |
| **Velocidad** | Mucho más rápida (nanosegundos). Permite lecturas/escrituras casi instantáneas. | Más lenta (micro‑ a milisegundos). Incluso los SSD, que son los más rápidos, son varios órdenes de magnitud más lentos que la RAM. |
| **Tipo de acceso** | Acceso **aleatorio** y prácticamente instantáneo a cualquier posición de memoria. | Acceso aleatorio también, pero 

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [11]:
# Leer API key desde Colab Secrets
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


In [15]:
# Definir la lista de preguntas
Preguntas = []

Preguntas.append("¿que es la inteligencia artificial?")
Preguntas.append("¿que es la nube?")
Preguntas.append("¿que es una maquina virtual?")

print(Preguntas)

['¿que es la inteligencia artificial?', '¿que es la nube?', '¿que es una maquina virtual?']


**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [32]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
import time

inicio = time.time()

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": Preguntas[0]}]
)
duracion = time.time() - inicio

print(response.choices[0].message.content)
print(f"Tiempo de respuesta: {duracion:.2f} segundos")
print("Tokens totales:", response.usage.total_tokens)

resultado_1 = {
    "Pregunta": Preguntas[0],
    "Respuesta": response.choices[0].message.content,
    "Tiempo de respuesta": duracion,
    "Tokens totales": response.usage.total_tokens
}


**Inteligencia Artificial (IA)** es la disciplina de la informática que se ocupa de crear sistemas capaces de realizar tareas que, normalmente, requerirían la inteligencia humana. Estas tareas incluyen el razonamiento, el aprendizaje, la percepción, la comprensión del lenguaje natural, la toma de decisiones y la resolución de problemas.

### Puntos clave

| Concepto | ¿Qué significa? |
|----------|-----------------|
| **Aprendizaje automático (Machine Learning)** | Algoritmos que permiten que un sistema mejore su desempeño con experiencia (datos). |
| **Redes neuronales artificiales** | Modelos inspirados en la estructura del cerebro que son la base de muchas aplicaciones modernas de IA (visión por computadora, reconocimiento de voz, etc.). |
| **Procesamiento del lenguaje natural (NLP)** | Capacidad de entender, generar y responder a texto o voz en lenguaje humano. |
| **Robótica** | Integración de IA en máquinas físicas que pueden interactuar con su entorno. |
| **IA explicable y éti

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [33]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
import time

inicio = time.time()

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": Preguntas[1]}]
)
duracion = time.time() - inicio

print(response.choices[0].message.content)
print(f"Tiempo de respuesta: {duracion:.2f} segundos")
print("Tokens totales:", response.usage.total_tokens)

resultado_1 = {
    "Pregunta": Preguntas[1],
    "Respuesta": response.choices[0].message.content,
    "Tiempo de respuesta": duracion,
    "Tokens totales": response.usage.total_tokens
}


**La “nube” (cloud en inglés) es un concepto de informática que permite acceder a recursos informáticos (almacenamiento, potencia de cálculo, software y servicios) a través de Internet, sin que tengas que poseer, instalar o gestionar físicamente esos recursos en tu propio equipo o en un centro de datos local.**

---

## 1. ¿Por qué se llama “nube”?
La palabra proviene de la imagen de una nube en el cielo: algo invisible, ubicuo y siempre accesible. Así como la lluvia llega a donde sea, la “nube” entrega datos y aplicaciones a cualquier dispositivo conectado a Internet.

---

## 2. Componentes clave

| Elemento | Descripción | Ejemplo |
|----------|-------------|---------|
| **Infraestructura como Servicio (IaaS)** | Proporciona máquinas virtuales, servidores, redes y almacenamiento virtuales. | Amazon EC2, Google Compute Engine, Microsoft Azure Virtual Machines |
| **Plataforma como Servicio (PaaS)** | Ofrece entornos de desarrollo, bases de datos, middleware y herramientas de desplieg

In [34]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
import time

inicio = time.time()

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": Preguntas[2]}]
)
duracion = time.time() - inicio

print(response.choices[0].message.content)
print(f"Tiempo de respuesta: {duracion:.2f} segundos")
print("Tokens totales:", response.usage.total_tokens)

resultado_1 = {
    "Pregunta": Preguntas[2],
    "Respuesta": response.choices[0].message.content,
    "Tiempo de respuesta": duracion,
    "Tokens totales": response.usage.total_tokens
}


Una **máquina virtual (VM)** es un software que emula un ordenador completo dentro de otro sistema operativo. Piensa en ella como una “computadora dentro de una computadora”.

### Cómo funciona
1. **Hipervisor (o Virtual Machine Monitor, VMM)**  
   - Es la capa de software que controla la VM. Puede ser **tipo 1** (nativo, corre directamente sobre el hardware, como VMware ESXi o Microsoft Hyper‑V) o **tipo 2** (hosteado, corre sobre un sistema operativo, como VirtualBox o VMware Workstation).

2. **Hardware virtualizado**  
   - El hipervisor asigna recursos del hardware real (CPU, RAM, disco, red) y los “traduzca” a la VM. Cada VM ve dispositivos virtuales (un disco duro virtual, una tarjeta de red virtual, etc.) como si fueran físicos.

3. **Sistema operativo invitado (guest OS)**  
   - Dentro de la VM se instala un SO (Windows, Linux, macOS, etc.) que funciona como si estuviera en una máquina real. El usuario interactúa con él mediante una consola gráfica o a través de comandos.

#

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [42]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []

resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)



**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [43]:
# Mostrar la tabla final de resultados
print(resultados)

[{'Pregunta': '¿que es una maquina virtual?', 'Respuesta': 'Una **máquina virtual (VM)** es un software que emula un ordenador completo dentro de otro sistema operativo. Piensa en ella como una “computadora dentro de una computadora”.\n\n### Cómo funciona\n1. **Hipervisor (o Virtual Machine Monitor, VMM)**  \n   - Es la capa de software que controla la VM. Puede ser **tipo 1** (nativo, corre directamente sobre el hardware, como VMware ESXi o Microsoft Hyper‑V) o **tipo 2** (hosteado, corre sobre un sistema operativo, como VirtualBox o VMware Workstation).\n\n2. **Hardware virtualizado**  \n   - El hipervisor asigna recursos del hardware real (CPU, RAM, disco, red) y los “traduzca” a la VM. Cada VM ve dispositivos virtuales (un disco duro virtual, una tarjeta de red virtual, etc.) como si fueran físicos.\n\n3. **Sistema operativo invitado (guest OS)**  \n   - Dentro de la VM se instala un SO (Windows, Linux, macOS, etc.) que funciona como si estuviera en una máquina real. El usuario int